<a href="https://colab.research.google.com/github/shin-noda/leetcode-neetcode-250/blob/main/Problem460_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Node:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.freq = 1
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self):
        # Sentinels for each frequency list
        self.head = Node(0, 0)
        self.tail = Node(0, 0)

        self.head.next = self.tail
        self.tail.prev = self.head
        self.size = 0


    def add_front(self, node):
        node.next = self.head.next
        node.prev = self.head

        self.head.next.prev = node
        self.head.next = node
        self.size += 1


    def remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev
        self.size -= 1


    def remove_tail(self):
        if self.size == 0:
            return None

        node = self.tail.prev
        self.remove(node)

        return node


class LFUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.size = 0
        self.min_freq = 0

        # key -> node
        self.cache = {}

        # freq -> DoublyLinkedList
        self.freq_map = {}


    def _update(self, node):
        # Remove from current frequency list
        freq = node.freq
        self.freq_map[freq].remove(node)

        # If this list is empty and it was the min_freq, increment min_freq
        if self.freq_map[freq].size == 0 and freq == self.min_freq:
            self.min_freq += 1

        # Increase frequency and add to new frequency list
        node.freq += 1
        new_freq = node.freq
        if new_freq not in self.freq_map:
            self.freq_map[new_freq] = DoublyLinkedList()

        self.freq_map[new_freq].add_front(node)


    def get(self, key):
        if key not in self.cache:
            return -1

        node = self.cache[key]
        self._update(node)

        return node.val


    def put(self, key, value):
        if self.capacity == 0:
            return

        if key in self.cache:
            node = self.cache[key]
            node.val = value
            self._update(node)

        else:
            if self.size == self.capacity:
                # Evict the LRU node from the min_freq list
                lfu_list = self.freq_map[self.min_freq]
                removed_node = lfu_list.remove_tail()
                self.cache.pop(removed_node.key)
                self.size -= 1

            # New node logic
            new_node = Node(key, value)
            self.cache[key] = new_node
            if 1 not in self.freq_map:
                self.freq_map[1] = DoublyLinkedList()

            self.freq_map[1].add_front(new_node)

            # Reset min_freq to 1 for new items
            self.min_freq = 1
            self.size += 1

In [ ]:
# Your LFUCache object will be instantiated and called as such:
# obj = LFUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)